# 13-5. 완료 감지·워커·재실행 관리 예제

## Goal

- 작업 상태 전이를 명시합니다.
- 부분 처리와 실행 실패를 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

메모리 상태만 사용하며 폴더 감시나 예약 실행은 수행하지 않습니다.


## Steps

### 작업 상태 기계

허용된 이전 상태에서만 다음 상태로 이동합니다.


In [1]:
ALLOWED_TRANSITIONS = {
    "waiting": {"ready", "failed"},
    "ready": {"running", "failed"},
    "running": {"complete", "partial", "failed"},
    "complete": set(),
    "partial": set(),
    "failed": set(),
}


def transition(job: dict, new_status: str, detail: str) -> dict:
    current = job["status"]
    if new_status not in ALLOWED_TRANSITIONS[current]:
        raise ValueError(f"허용하지 않은 상태 전이: {current} -> {new_status}")
    history = [*job.get("history", []), {"from": current, "to": new_status, "detail": detail}]
    return {**job, "status": new_status, "history": history}


job = {"id": "SYNTHETIC-001", "status": "waiting", "history": []}
job = transition(job, "ready", "manifest 저장 완료")
job = transition(job, "running", "worker-01이 작업 확보")
job = transition(job, "partial", "입력 한 건 누락")
print(job)


{'id': 'SYNTHETIC-001', 'status': 'partial', 'history': [{'from': 'waiting', 'to': 'ready', 'detail': 'manifest 저장 완료'}, {'from': 'ready', 'to': 'running', 'detail': 'worker-01이 작업 확보'}, {'from': 'running', 'to': 'partial', 'detail': '입력 한 건 누락'}]}


## Checks

완료 상태의 재실행과 잘못된 건너뛰기를 거부합니다.


In [2]:
assert job["status"] == "partial"
assert len(job["history"]) == 3
try:
    transition(job, "running", "같은 결과 재사용")
except ValueError:
    print("종료 상태 재사용 거부 확인")


종료 상태 재사용 거부 확인


## Next Steps

재실행은 새 출력 디렉터리와 새 실행 ID를 사용하고 이전 완료 표시를 덮어쓰지 않습니다.
